In [1]:
# General imports
import numpy as np
import scipy as sp
from scipy.optimize import minimize
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Lambda
from tensorflow.python.keras import backend as K
from tensorflow.keras.optimizers import Adam

# Neurophox imports
from neurophox.numpy import *
from neurophox.tensorflow import *
from neurophox.ml import LinearMultiModelRunner
from neurophox.ml.linear import complex_mse
from neurophox.ml.nonlinearities import cnorm, cnormsq
from neurophox.initializers import *
from neurophox.components import *
from neurophox.helpers import *

import pyswarms as ps
import copy

In [2]:
plt.rcParams['text.usetex'] = True 
plt.rcParams['text.latex.preamble'] = r"\usepackage{siunitx} \usepackage{amsmath} \usepackage{sansmathfonts} \usepackage[T1]{fontenc} \renewcommand*\familydefault{\sfdefault}"

In [3]:
def clements_decomposition_working(u: np.ndarray, pbar_handle: Callable = None, smmzi: bool = False) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Clements decomposition of unitary matrix :math:`U` to output the required phases for producing the matrix

    Args:
        u: unitary matrix :math:`U` to be decomposed into pairwise operators.
        pbar_handle: Useful for larger matrices

    Returns:
        The phases $(\theta, \phi, \gamma)$ layer that outputs the unitary :math:`U`

    """
    u_hat = u.T.copy()
    n = u.shape[0]
    # odd and even layer dimensions
    theta_checkerboard = np.zeros_like(u, dtype=NP_FLOAT)
    phi_checkerboard = np.zeros_like(u, dtype=NP_FLOAT)
    phi_checkerboard = np.hstack((np.zeros((n, 1)), phi_checkerboard))
    iterator = pbar_handle(range(n - 1)) if pbar_handle else range(n - 1)
    MZI = SMMZI if smmzi else BlochMZI
    for i in iterator:
        if i % 2:
            for j in range(i + 1):
                pairwise_index = n + j - i - 2
                target_row, target_col = n + j - i - 1, j
                theta = np.arctan(np.abs(u_hat[target_row - 1, target_col] / u_hat[target_row, target_col])) * 2
                phi = np.angle(u_hat[target_row, target_col] / u_hat[target_row - 1, target_col])
                mzi = MZI(theta, phi, hadamard=False, dtype=np.complex128)
                left_multiplier = mzi.givens_rotation(units=n, m=pairwise_index)
                u_hat = left_multiplier @ u_hat
                theta_checkerboard[pairwise_index, j] = theta
                phi_checkerboard[pairwise_index, j] = -phi + np.pi
                phi_checkerboard[pairwise_index + 1, j] = np.pi
        else:
            for j in range(i + 1):
                pairwise_index = i - j
                target_row, target_col = n - j - 1, i - j
                theta = np.arctan(np.abs(u_hat[target_row, target_col + 1] / u_hat[target_row, target_col])) * 2
                phi = np.angle(-u_hat[target_row, target_col] / u_hat[target_row, target_col + 1])
                mzi = BlochMZI(theta, phi, hadamard=False, dtype=np.complex128)
                right_multiplier = mzi.givens_rotation(units=n, m=pairwise_index)
                u_hat = u_hat @ right_multiplier.conj().T
                theta_checkerboard[pairwise_index, -j - 1] = theta
                phi_checkerboard[pairwise_index, -j - 1] = phi + np.pi

    diag_phases = np.angle(np.diag(u_hat))
    theta = checkerboard_to_param(np.fliplr(theta_checkerboard), n)
    phi_checkerboard = np.fliplr(phi_checkerboard)
    if n % 2:
        phi_checkerboard[:, :-1] += np.fliplr(np.diag(diag_phases))
    else:
        phi_checkerboard[:, 1:] += np.fliplr(np.diag(diag_phases))
        phi_checkerboard[-1, 2::2] += np.pi / 2  # neurophox layers assume pi / 2 phase shift in even layer "bounces"
        phi_checkerboard[0, 2::2] += np.pi / 2

    gamma = phi_checkerboard[:, 0]
    external_phases = phi_checkerboard[:, 1:]
    phi, gamma = grid_common_mode_flow(external_phases, gamma=gamma)
    phi = checkerboard_to_param(phi, n)

    # I don't understand this portion
    gamma_adj = np.zeros_like(gamma)
    gamma_adj[1::4] = 1
    gamma_adj[2::4] = 1
    gamma += np.pi * (1 - gamma_adj) if (n // 2) % 2 else np.pi * gamma_adj
    gamma = np.mod(gamma, 2 * np.pi)

    return theta, phi, gamma

def checkerboard_to_param(checkerboard: np.ndarray, units: int):
    param = np.zeros((units, units // 2))
    if units % 2:
        param[::2, :] = checkerboard.T[::2, :-1:2]
    else:
        param[::2, :] = checkerboard.T[::2, ::2]
    param[1::2, :] = checkerboard.T[1::2, 1::2]
    return param

def grid_common_mode_flow(external_phases: np.ndarray, gamma: np.ndarray, basis: str = "sm"):
    """In a grid mesh (e.g., triangular, rectangular meshes), arrange phases according to single-mode (:code:`sm`),
       differential mode (:code:`diff`), or max-:math:`\\pi` (:code:`maxpi`, all external phase shifts are at most
       :math:`\\pi`). This is done using a procedure called "common mode flow" where common modes are shifted
       throughout the mesh until phases are correctly set.

    Args:
        external_phases: external phases in the grid mesh
        gamma: input phase shifts
        basis: single-mode (:code:`sm`), differential mode (:code:`diff`), or max-:math:`\\pi` (:code:`maxpi`)

    Returns:
        new external phases shifts and new gamma resulting

    """
    units, num_layers = external_phases.shape
    phase_shifts = np.hstack((gamma[:, np.newaxis], external_phases)).T
    new_phase_shifts = np.zeros_like(external_phases.T)

    for i in range(num_layers):
        current_layer = num_layers - i
        start_idx = (current_layer - 1) % 2
        end_idx = units - (current_layer + units - 1) % 2
        # calculate phase information
        upper_phase = phase_shifts[current_layer][start_idx:end_idx][::2]
        lower_phase = phase_shifts[current_layer][start_idx:end_idx][1::2]
        upper_phase = np.mod(upper_phase, 2 * np.pi)
        lower_phase = np.mod(lower_phase, 2 * np.pi)
        if basis == "sm":
            new_phase_shifts[-i - 1][start_idx:end_idx][::2] = upper_phase - lower_phase
        # assign differential phase to the single mode layer and keep common mode layer
        else:
            phase_diff = upper_phase - lower_phase
            phase_diff[phase_diff > np.pi] -= 2 * np.pi
            phase_diff[phase_diff < -np.pi] += 2 * np.pi
            if basis == "diff":
                new_phase_shifts[-i - 1][start_idx:end_idx][::2] = phase_diff / 2
                new_phase_shifts[-i - 1][start_idx:end_idx][1::2] = -phase_diff / 2
            elif basis == "pimax":
                new_phase_shifts[-i - 1][start_idx:end_idx][::2] = phase_diff * (phase_diff >= 0)
                new_phase_shifts[-i - 1][start_idx:end_idx][1::2] = -phase_diff * (phase_diff < 0)
        # update the previous layer with the common mode calculated for the current layer\
        phase_shifts[current_layer] -= new_phase_shifts[-i - 1]
        phase_shifts[current_layer - 1] += np.mod(phase_shifts[current_layer], 2 * np.pi)
        phase_shifts[current_layer] = 0
    new_gamma = np.mod(phase_shifts[0], 2 * np.pi)
    return np.mod(new_phase_shifts.T, 2 * np.pi), new_gamma

In [4]:
def NormalPhaseInitializer(layer: RM, sigma : float):
    """Initialises a mesh layer with phases from a normal distribution 

    Args:
        layer (RM): Neurophox rectangular mesh layer
        sigma (float): Standard deviation of the phase distribution
    """
    theta_b = (2 * np.pi * abs(np.random.normal(0, sigma, layer.theta.shape))) % np.pi
    theta_b[1::2, -1] = 0

    phi_b = (2 * np.pi * abs(np.random.normal(0, sigma, layer.phi.shape))) % (2 * np.pi)
    phi_b[1::2, -1] = 0

    gamma_b = (2 * np.pi * abs(np.random.normal(0, sigma, layer.gamma.shape))) % (2 * np.pi)

    t_ = tf.convert_to_tensor(theta_b, dtype=tf.float32)
    p_ = tf.convert_to_tensor(phi_b, dtype=tf.float32)
    g_ = tf.convert_to_tensor(gamma_b.reshape(1, -1), dtype=tf.float32)
    layer.theta.assign(t_)
    layer.phi.assign(p_)
    layer.gamma.assign(g_)
    return theta_b, phi_b, gamma_b

## Particle Swarm Optimisation:

In [11]:
def objective_function(params, target_matrix, N, layer):    
    n_particles = params.shape[0]
    losses = np.zeros(n_particles)

    for i in range(n_particles):
        A = compute_complex_matrix(params[i], N, layer)

        # Calculate the Frobenius norm (or another suitable norm)
        losses[i] = np.linalg.norm(target_matrix - A, 'fro') / N
    
    return losses

def compute_complex_matrix(params, N, layer):
    M = (N * (N-1)) // 2
    t, p, g = params[:M], params[M:2*M], params[2*M:]
    m = layer.phases.mask.astype(bool)
    theta, phi, _ = layer.phases.params
    np.putmask(theta, m, t)
    np.putmask(phi, m, p)
    
    PhaseInitializer(layer, theta, phi, g)

    return layer.matrix

def PhaseInitializer(layer: RM, theta: np.ndarray, phi: np.ndarray, gamma: np.ndarray):
    """Initialises a mesh layer with the desired phases

    Args:
        layer (RM): Neurophox rectangular mesh layer
        theta (np.ndarray): Theta
        phi (np.ndarray): Phi
        gamma (np.ndarray): Gamma
    """
    t_ = tf.convert_to_tensor(theta, dtype=tf.float32)
    p_ = tf.convert_to_tensor(phi, dtype=tf.float32)
    g_ = tf.convert_to_tensor(gamma.reshape(1, -1), dtype=tf.float32)
    layer.theta.assign(t_)
    layer.phi.assign(p_)
    layer.gamma.assign(g_)
    # print('Phases have been initialized successfully.')

In [12]:
def optimize(N, target_matrix, layer, n_particles=50, iters=1000, c1=0.5, c2=0.3, w=0.9, bound_factor=0.5):
    # Dimension of each particle: N^2 
    dimensions = N**2
    M = (N*(N-1)) // 2
    
    # Bounds for the particles: different for theta and phi
    ul = np.zeros(dimensions)
    ul[:M] = np.pi * bound_factor
    ul[M:] = 2 * np.pi * bound_factor
    bounds = (np.zeros(dimensions), ul)
    
    # Create a PSO optimizer instance
    options = {'c1': c1, 'c2': c2, 'w': w}
    optimizer = ps.single.GlobalBestPSO(n_particles=n_particles, dimensions=dimensions, options=options, bounds=bounds)
    
    # Run the optimizer
    cost, pos = optimizer.optimize(objective_function, iters=iters, N=N, target_matrix=target_matrix, layer=layer)
    return cost, pos

In [7]:
U = RM(6)
phase_errors = NormalPhaseInitializer(U, 0.05)
target_matrix = U.matrix
phase_errors

(array([[0.43622977, 0.1744861 , 0.31615084],
        [0.28253481, 0.63487765, 0.        ],
        [0.03562729, 0.69975311, 0.44519016],
        [0.07525082, 0.18815767, 0.        ],
        [0.40824502, 0.46883857, 0.09311703],
        [0.08303871, 0.03783032, 0.        ]]),
 array([[0.16790363, 0.29792133, 0.35454106],
        [0.10920538, 0.07475821, 0.        ],
        [0.08440019, 0.1788903 , 0.10213272],
        [0.39166003, 0.2132249 , 0.        ],
        [0.40217612, 0.10926365, 0.01728935],
        [0.21381686, 0.44398682, 0.        ]]),
 array([[0.06532597, 0.05922675, 0.25661258, 0.4528045 , 0.1715957 ,
         0.68541   ]]))

In [18]:
layer = RM(6)

cost2, pos2 = optimize(6, target_matrix, layer, 100, 200, 0.4, 0.6, 0.95, 0.2)
cost3, pos3 = optimize(6, target_matrix, layer, 100, 200, 0.6, 0.6, 0.95, 0.2)


2024-06-12 14:15:32,615 - pyswarms.single.global_best - INFO - Optimize for 200 iters with {'c1': 0.4, 'c2': 0.6, 'w': 0.95}
pyswarms.single.global_best: 100%|██████████|200/200, best_cost=0.203
2024-06-12 14:17:45,174 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.20294090112050375, best pos: [0.07633635 0.06795432 0.50073352 0.22889666 0.34057384 0.34825276
 0.0725846  0.36145514 0.31798108 0.18704065 0.58231119 0.06943249
 0.12436677 0.39599593 0.20915923 0.15022181 0.1169528  0.2227701
 0.66320119 0.29875723 0.22398594 0.05384233 0.44736229 0.45915582
 0.51459244 0.02876058 0.33697085 0.57099784 0.15464506 0.26052529
 0.55584584 0.21987202 0.34079808 0.54076493 0.79296427 0.11625943]
2024-06-12 14:17:45,224 - pyswarms.single.global_best - INFO - Optimize for 200 iters with {'c1': 0.6, 'c2': 0.6, 'w': 0.95}
pyswarms.single.global_best: 100%|██████████|200/200, best_cost=0.173
2024-06-12 14:19:55,635 - pyswarms.single.global_best - INFO - Optimization fin

In [ ]:
phase_errors, pos2

((array([[0.05765   , 0.24441689, 0.35181395],
         [0.52618172, 0.44151763, 0.        ],
         [0.26306957, 0.81483029, 0.05590008],
         [0.20583313, 0.11575768, 0.        ],
         [0.25018075, 0.34506157, 0.75306541],
         [0.41444678, 0.6024447 , 0.        ]]),
  array([[0.68080051, 0.29046165, 0.09628802],
         [0.10509366, 0.59835301, 0.        ],
         [0.57552637, 0.45419875, 0.23119783],
         [0.63647099, 0.16146716, 0.        ],
         [0.25635472, 0.01946547, 0.33299218],
         [1.11353661, 0.00941957, 0.        ]]),
  array([[0.37298763, 0.07307545, 0.32847638, 0.20454265, 0.28979431,
          0.42669036]])),
 array([0.13381014, 0.35237022, 0.31875706, 0.30962311, 0.37893753,
        0.61642528, 0.37027112, 0.57002529, 0.04193021, 0.16042577,
        0.2284259 , 0.26898543, 0.40319022, 0.32516016, 0.58084365,
        0.4531599 , 0.14516537, 0.50107663, 0.22162117, 0.49547195,
        0.40944797, 0.48878213, 0.32429834, 0.36138609, 0.468697

In [ ]:
compute_complex_matrix(pos2, 6, layer)

array([[-7.77904391e-02+0.09984053j,  5.80524094e-04-0.05936633j,
         8.14948082e-02+0.12283748j, -3.08562778e-02-0.02512673j,
         1.49238870e-01+0.13336746j,  8.56234729e-01+0.42888796j],
       [-2.05137283e-02-0.16207014j,  4.44412865e-02+0.1795437j ,
        -8.62372816e-02-0.12450631j,  6.51103333e-02+0.05281711j,
         7.66014099e-01+0.54951346j, -1.28734186e-01-0.06179486j],
       [-8.55163336e-02+0.06685636j,  4.29603383e-02-0.20739488j,
         2.65345685e-02+0.12174575j,  5.68713129e-01+0.7769647j ,
        -2.52837762e-02+0.00506629j,  7.74440065e-04-0.00763932j],
       [ 4.05388921e-02-0.19448146j,  2.81878989e-02+0.18496805j,
         3.84130239e-01+0.8671494j , -3.64376158e-02-0.05282464j,
         4.90499586e-02+0.04675044j, -9.47783217e-02-0.09126441j],
       [-3.18534002e-02+0.03207233j, -1.03857383e-01+0.93174285j,
        -1.12126153e-02-0.13215944j,  1.45195156e-01+0.1610414j ,
        -1.13202065e-01-0.16350986j,  9.48296413e-02+0.07664847j],
     